In [1]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
import os
# os.chdir("../")

c:\Users\gaura\OneDrive\Desktop\Learn\1learn\MedBot_RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def load_pdfs(pdf_directory):
    # Load PDFs from the specified directory
    loader = DirectoryLoader(pdf_directory, glob="*.pdf", show_progress=True, loader_cls=PyPDFLoader)
    documents = loader.load()
    return documents

In [3]:
dataset = load_pdfs("data")

FileNotFoundError: Directory not found: 'data'

In [ ]:
dataset[1]

Document(metadata={'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'creator': 'PyPDF', 'creationdate': '2004-12-18T17:00:02-05:00', 'moddate': '2004-12-18T16:15:31-06:00', 'source': 'data\\Medical_book.pdf', 'total_pages': 637, 'page': 1, 'page_label': '2'}, page_content='The GALE\nENCYCLOPEDIA\nof MEDICINE\nSECOND EDITION')

In [60]:
from typing import List
from langchain.schema import Document

def filter_data(documents: List[Document]) -> List[Document]:
    # Filter documents based on specific criteria (e.g., keywords, metadata)
    filtered_documents: List[Document] = []
    for doc in documents:
        src = doc.metadata.get("source", "")
        page_num = doc.metadata.get("page_label", "")
        # page_num = doc.metadata.get("page_number")
        filtered_documents.append(
            Document(
                page_content=doc.page_content,
                metadata={"source": src, "page_label": page_num}
            )
        )
    return filtered_documents

In [61]:
filered_dataset = filter_data(dataset)
filered_dataset[1]

Document(metadata={'source': 'data\\Medical_book.pdf', 'page_label': '2'}, page_content='The GALE\nENCYCLOPEDIA\nof MEDICINE\nSECOND EDITION')

### Chunking

In [62]:
def split_data(filered_dataset):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    text_chunks = text_splitter.split_documents(filered_dataset)
    return text_chunks

In [63]:
text_chunks = split_data(filered_dataset)
text_chunks[1]

Document(metadata={'source': 'data\\Medical_book.pdf', 'page_label': '3'}, page_content='The GALE\nENCYCLOPEDIA\nof MEDICINE\nSECOND EDITION\nJACQUELINE L. LONGE, EDITOR\nDEIRDRE S. BLANCHFIELD, ASSOCIATE EDITOR\nVOLUME\nA-B\n1')

### Embeddings

In [13]:
# from langchain_huggingface.embeddings import HuggingFaceEmbeddings
# def create_embeddings(text_chunks):
#     embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
#     texts = [chunk.page_content for chunk in text_chunks]
#     embeddings = embedding_model.embed_documents(texts)
#     return embeddings

# embeddings = create_embeddings(text_chunks)
from langchain.embeddings import HuggingFaceEmbeddings
import torch

def embedding_model():
    model_nmae = "sentence-transformers/all-MiniLM-L6-v2"
    embedding_model = HuggingFaceEmbeddings(model_name=model_nmae,
                                            model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"})
    return embedding_model

embedding_model = embedding_model()

### Pinecone DB

In [14]:
from pinecone import Pinecone
from dotenv import load_dotenv
import os

load_dotenv() 
pinecone_key = os.getenv("PINECONE_API_KEY")

# Initialize the Pinecone client
pc_client = Pinecone(api_key=pinecone_key)

In [16]:
from pinecone import ServerlessSpec

index_name = "medbot-rag"

if not pc_client.has_index(index_name):
    pc_client.create_index(
        name=index_name,
        dimension=384,  # Dimension of the embeddings
        metric="cosine",  # Similarity metric
        spec=ServerlessSpec(cloud="aws", region="us-east-1")  
    )
index = pc_client.Index(index_name)

In [ ]:
from langchain_pinecone import PineconeVectorStore

addocs = PineconeVectorStore.from_documents(
    documents=text_chunks,     
    embedding=embedding_model,        
    index_name=index_name      
)

In [ ]:
# add new documents to the existing index
newdoc = Document(page_content="This is a new document to be added to the index.", metadata={"source": "new_doc.txt"})
addocs.add_documents([newdoc])

### Retriever

In [20]:
retrive = addocs.as_retriever(search_type = "similarity",search_kwargs={"k": 3})

query = "What are the symptoms of diabetes?"
retrive.get_relevant_documents(query)

[Document(id='5e2d7d30-6cd9-40b9-8f6f-8a5ac42f3a38', metadata={'source': 'data\\Medical_book.pdf'}, page_content='affected and can range greatly.\n• Type I diabetes mellitus. Characterized by fatigue and\nan abnormally high level of glucose in the blood\n(hyperglycemia).\n• Amyotrophic lateral schlerosis. First signs are stum-\nbling and difficulty climbing stairs. Later, muscle\ncramps and twitching may be observed as well as\nweakness in the hands making fastening buttons or\nturning a key difficult. Speech may become slowed or\nslurred. There may also be difficluty swallowing. As'),
 Document(id='2a0cf53c-9a34-42ed-871c-33104c32046d', metadata={'source': 'data\\Medical_book.pdf'}, page_content='begin to fall. A person with diabetes mellitus either does\nnot make enough insulin, or makes insulin that does not\nwork properly. The result is blood sugar that remains\nhigh, a condition called hyperglycemia.\nDiabetes must be diagnosed as early as possible. If\nleft untreated, it can dama

In [23]:
from openai import OpenAI

load_dotenv(override=True)
api_key_hf = os.getenv('HF_TOKEN')

MODEL = 'Qwen/Qwen2.5-Coder-3B-Instruct:nscale'
# openai = OpenAI(
#     base_url="https://router.huggingface.co/v1",
#     api_key = os.getenv('HF_TOKEN'))


In [ ]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

MODEL_NAME = 'Qwen/Qwen2.5-Coder-3B-Instruct'

llm_endpoint = HuggingFaceEndpoint(
    repo_id=MODEL_NAME,
    max_new_tokens=100,
    temperature=0.1,
    huggingfacehub_api_token=api_key_hf,
    provider="nscale",
    task="conversational",
)

llm = ChatHuggingFace(llm=llm_endpoint)

In [53]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain 
from langchain_core.prompts import ChatPromptTemplate

system_prompt="""
            You are an experienced medical assistant for answering questions related to medical information. 
            Use the retrieved context information to provide accurate and concise answers to the user's queries.
            if you don't know the answer, say you don't know. Do not try to make up an answer. keep the answer concise and to the point.:
            \n\n
            "{context}
    """
prompt = ChatPromptTemplate.from_messages([
        ("system", system_prompt),
        ("human", "{input}")
])


In [54]:
qa_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retrive, qa_chain)

In [55]:
response = rag_chain.invoke({"input": "What are the symptoms of diabetes?"})
print(response["answer"])

Signs of diabetes often include frequent urination, increased thirst, excessive hunger, unexplained weight loss or gain, fatigue, blurred vision, and sores that do not heal. It's important for anyone who frequently experiences these symptoms to see a doctor for proper diagnosis and management.
